# LeetCode 1375 — Number of Times Binary String Is Prefix-Aligned bằng Python

Link đề: https://leetcode.com/problems/number-of-times-binary-string-is-prefix-aligned/ · Độ khó: **Medium** · Chủ đề: Array (nhận dạng: prefix max, hoán vị)

Skeleton LeetCode:

```python
class Solution:
    def numTimesAllBlue(self, flips: list[int]) -> int:
        ...
```

(Tên hàm `numTimesAllBlue` đến từ phiên bản cũ của đề: "bóng đèn chuyển xanh".)

## 1. Mục tiêu học

1. Mô phỏng chính xác quá trình bật bit và hiểu định nghĩa **prefix-aligned** (thẳng hàng tiền tố).
2. Nhận ra tính chất của **hoán vị**: sau bước `i` có đúng `i` bit 1 ⇒ "prefix-aligned" ⇔ **mọi vị trí đã bật đều ≤ `i`** ⇔ `max(flips[0..i-1]) == i`.
3. Biết thêm một cách nhìn khác: **tổng** của `i` số nguyên dương phân biệt bằng `i(i+1)/2` ⇔ chúng là `{1..i}`.
4. Cải tiến từ `O(n²)` mô phỏng → `O(n)` với `O(1)` bộ nhớ; biết kỹ thuật **con trỏ khấu hao** ("vị trí 0 đầu tiên").
5. Cẩn thận với **chỉ số 1-based** của đề khi viết code Python 0-based.

## 2. Đề bài & ví dụ

Có một chuỗi nhị phân độ dài `n`, đánh số **từ 1**, ban đầu toàn bit 0. Ta bật lần lượt từng bit thành 1: ở bước thứ `i` (1-based), bật bit tại vị trí `flips[i]`. Mảng `flips` là một **hoán vị** của `1..n` (mỗi vị trí được bật đúng một lần).

Chuỗi được gọi là **prefix-aligned** sau bước `i` nếu các bit trong đoạn `[1, i]` đều là 1 và mọi bit khác đều là 0. Đếm số bước mà sau bước đó chuỗi prefix-aligned.

- **Ví dụ 1:** `flips = [3,2,4,1,5]` → `2`. Các trạng thái: `00100`, `01100`, `01110`, `11110` ✓, `11111` ✓.
- **Ví dụ 2:** `flips = [4,1,2,3]` → `1`. Các trạng thái: `0001`, `1001`, `1101`, `1111` ✓.

**Ràng buộc:** `n == flips.length`; `1 <= n <= 5·10^4`; `flips` là hoán vị của `[1, n]`.

In [1]:
import random
import time

EXAMPLES = [([3, 2, 4, 1, 5], 2), ([4, 1, 2, 3], 1)]
for flips, want in EXAMPLES:
    print(f"flips = {flips} -> mong đợi {want}")

flips = [3, 2, 4, 1, 5] -> mong đợi 2
flips = [4, 1, 2, 3] -> mong đợi 1


## 3. Kiến thức nền

### 3.1 Chỉ số 1-based vs 0-based

Đề đánh số bit và bước từ 1. Trong Python, list đánh số từ 0. Hai cách phổ biến:
- Tạo mảng dài `n + 1` và bỏ ô 0 không dùng.
- Dùng `enumerate(flips, start=1)` để có số bước 1-based.

In [2]:
flips = [3, 2, 4, 1, 5]
for step, pos in enumerate(flips, start=1):
    print(f"bước {step}: bật vị trí {pos}")
bits = [0] * (len(flips) + 1)     # ô 0 bỏ trống
print("mảng bit (bỏ ô 0):", bits[1:])

bước 1: bật vị trí 3
bước 2: bật vị trí 2
bước 3: bật vị trí 4
bước 4: bật vị trí 1
bước 5: bật vị trí 5
mảng bit (bỏ ô 0): [0, 0, 0, 0, 0]


### 3.2 `all()` và cắt lát

`all(iterable)` trả `True` nếu mọi phần tử đúng (và `True` với dãy rỗng). `bits[1:i+1]` là các bit từ vị trí 1 tới `i`.

In [3]:
bits = [0, 1, 1, 0, 1]          # bỏ ô 0: vị trí 1..4 = 1,1,0,1
print(all(bits[1:3]), all(bits[1:4]), all([]))

True False True


### 3.3 Hoán vị và cực đại tiền tố (prefix max)

Một **hoán vị** của `1..n` chứa mỗi số đúng một lần. Cực đại tiền tố `M[i] = max(a[0..i-1])` cập nhật trong `O(1)` mỗi bước: `M = max(M, a[i])`.

Tính chất then chốt: **`i` số phân biệt trong `1..n` có max bằng `i` khi và chỉ khi chúng chính là `{1, 2, ..., i}`** (vì `i` số phân biệt cùng nằm trong `[1, i]` thì phải lấp đầy `[1, i]`).

In [4]:
a = [3, 2, 4, 1, 5]
mx = 0
for i, x in enumerate(a, start=1):
    mx = max(mx, x)
    print(f"i={i}: tập đã thấy = {sorted(a[:i])}, max = {mx}, max == i? {mx == i}")

i=1: tập đã thấy = [3], max = 3, max == i? False
i=2: tập đã thấy = [2, 3], max = 3, max == i? False
i=3: tập đã thấy = [2, 3, 4], max = 4, max == i? False
i=4: tập đã thấy = [1, 2, 3, 4], max = 4, max == i? True
i=5: tập đã thấy = [1, 2, 3, 4, 5], max = 5, max == i? True


### 3.4 Tổng dãy 1..i

`1 + 2 + ... + i = i(i+1)/2`. Và tổng của `i` số nguyên dương **phân biệt** luôn `≥ i(i+1)/2`, dấu bằng **chỉ khi** đó là `{1..i}`. Đây là nền cho Cách 3.

In [5]:
i = 4
print("tổng 1..4 =", i * (i + 1) // 2)
print("tổng {1,2,3,5} =", sum([1, 2, 3, 5]), "> 10")
print("tổng {1,2,4,3} =", sum([1, 2, 4, 3]), "== 10")

tổng 1..4 = 10
tổng {1,2,3,5} = 11 > 10
tổng {1,2,4,3} = 10 == 10


## 4. Ý tưởng nền tảng / trực giác

Tính tay `flips = [3,2,4,1,5]`:

| bước i | bật | chuỗi | các vị trí đã bật | max | prefix-aligned? |
|---|---|---|---|---|---|
| 1 | 3 | 00100 | {3} | 3 | không (3 > 1) |
| 2 | 2 | 01100 | {2,3} | 3 | không (3 > 2) |
| 3 | 4 | 01110 | {2,3,4} | 4 | không |
| 4 | 1 | 11110 | {1,2,3,4} | 4 | **có** (4 == 4) |
| 5 | 5 | 11111 | {1..5} | 5 | **có** |

Quan sát: sau bước `i` luôn có **đúng `i` bit 1** (mỗi bước bật một bit mới). Vậy chuỗi prefix-aligned ⇔ `i` bit 1 nằm gọn trong `[1, i]` ⇔ **không bit nào bật ở vị trí > `i`** ⇔ `max ≤ i`. Mà max của `i` số phân biệt ≥ `i`, nên điều kiện là `max == i`.

Hình dung bằng "cửa sổ": sau bước `i`, vẽ dấu `|` sau vị trí `i`. Chuỗi aligned ⇔ **không có bit 1 nào nằm bên phải dấu `|`**. Hàm dưới in trực quan điều này cho Ví dụ 2 — chú ý bit ở vị trí 4 nằm ngoài cửa sổ suốt 3 bước đầu, nên chỉ khi cửa sổ mở tới 4 thì mới aligned.

In [6]:
def show_window(flips):
    n = len(flips)
    bits = ["0"] * n
    for i, pos in enumerate(flips, start=1):
        bits[pos - 1] = "1"
        inside, outside = "".join(bits[:i]), "".join(bits[i:])
        ok = "1" not in outside            # không có bit 1 ngoài cửa sổ
        line = inside + "|" + outside
        print(f"bước {i}: {line}  -> {'ALIGNED' if ok else 'chưa'} (số bit 1 ngoài cửa sổ: {outside.count('1')}, max đã bật = {max(flips[:i])})")

show_window([4, 1, 2, 3])
print()
show_window([2, 1, 4, 3, 5])

bước 1: 0|001  -> chưa (số bit 1 ngoài cửa sổ: 1, max đã bật = 4)
bước 2: 10|01  -> chưa (số bit 1 ngoài cửa sổ: 1, max đã bật = 4)
bước 3: 110|1  -> chưa (số bit 1 ngoài cửa sổ: 1, max đã bật = 4)
bước 4: 1111|  -> ALIGNED (số bit 1 ngoài cửa sổ: 0, max đã bật = 4)

bước 1: 0|1000  -> chưa (số bit 1 ngoài cửa sổ: 1, max đã bật = 2)
bước 2: 11|000  -> ALIGNED (số bit 1 ngoài cửa sổ: 0, max đã bật = 2)
bước 3: 110|10  -> chưa (số bit 1 ngoài cửa sổ: 1, max đã bật = 4)
bước 4: 1111|0  -> ALIGNED (số bit 1 ngoài cửa sổ: 0, max đã bật = 4)
bước 5: 11111|  -> ALIGNED (số bit 1 ngoài cửa sổ: 0, max đã bật = 5)


Vì sau bước `i` luôn có đúng `i` bit 1 và cửa sổ có đúng `i` ô, "không có bit 1 ngoài cửa sổ" tương đương "cửa sổ toàn 1". Đây là lý do ta chỉ cần theo dõi **bit 1 xa nhất** (max) thay vì cả chuỗi.

## 5. Cách 1 — Brute Force / baseline: mô phỏng và kiểm tra tiền tố

**Ý tưởng:** giữ mảng bit; sau mỗi bước `i`, kiểm tra trực tiếp `bits[1..i]` toàn 1 và `bits[i+1..n]` toàn 0.

In [7]:
def count_brute(flips):
    n = len(flips)
    bits = [0] * (n + 1)
    ans = 0
    for i, pos in enumerate(flips, start=1):
        bits[pos] = 1
        if all(bits[1:i + 1]) and not any(bits[i + 1:]):
            ans += 1
    return ans

for flips, want in EXAMPLES:
    print(flips, "->", count_brute(flips), "(mong đợi", want, ")")

[3, 2, 4, 1, 5] -> 2 (mong đợi 2 )
[4, 1, 2, 3] -> 1 (mong đợi 1 )


**Độ phức tạp:** mỗi bước kiểm tra `O(n)` ⇒ tổng `O(n²)`; bộ nhớ `O(n)`. Với `n = 5·10^4` là `2,5·10^9` phép — quá chậm trong Python.

(Cắt lát và `all/any` chạy bằng C nên trông nhanh với `n` nhỏ, nhưng thời gian vẫn tăng **bậc hai** — gấp đôi `n` thì thời gian tăng khoảng 4 lần, xem số đo bên dưới.)

**Nút thắt:** ta kiểm tra lại toàn bộ chuỗi sau mỗi bước, dù chỉ **một bit** thay đổi. Cần một đại lượng "tóm tắt" cập nhật được trong `O(1)`.

In [8]:
rng = random.Random(0)
for n in [4000, 8000, 16000]:
    p = list(range(1, n + 1)); rng.shuffle(p)
    t0 = time.perf_counter()
    count_brute(p)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=4000: 0.008s
n=8000: 0.030s
n=16000: 0.117s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** sau bước `i` có đúng `i` bit 1 tại các vị trí `flips[0..i-1]` (phân biệt). Do đó:

> **prefix-aligned sau bước `i` ⇔ `max(flips[0..i-1]) == i`.**

**Quy tắc "nếu… thì…":**
- Nếu bit vừa bật làm `max` vượt `i` ⇒ chưa thể aligned cho tới khi `i` đuổi kịp `max`.
- Nếu `max == i` ⇒ mọi vị trí `1..i` đều đã bật (chuồng bồ câu) ⇒ đếm.

**Invariant:** sau khi xử lý bước `i`, biến `mx` = giá trị lớn nhất trong `i` vị trí đầu tiên của `flips`.

Các đại lượng tóm tắt khác cũng dùng được: **tổng** (`== i(i+1)/2`), hoặc **vị trí bit 0 đầu tiên** (`== i + 1`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cực đại tiền tố: `max == i` (khuyên dùng)

In [9]:
def count_max(flips):
    mx = ans = 0
    for i, pos in enumerate(flips, start=1):
        mx = max(mx, pos)       # cập nhật TRƯỚC khi so sánh
        if mx == i:
            ans += 1
    return ans

for flips, want in EXAMPLES:
    print(flips, "->", count_max(flips), "(mong đợi", want, ")")

[3, 2, 4, 1, 5] -> 2 (mong đợi 2 )
[4, 1, 2, 3] -> 1 (mong đợi 1 )


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def numTimesAllBlue(self, flips: list[int]) -> int:
        mx = ans = 0
        for i, pos in enumerate(flips, 1):
            if pos > mx:
                mx = pos
            if mx == i:
                ans += 1
        return ans

print([Solution().numTimesAllBlue(f) for f, _ in EXAMPLES])

[2, 1]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — So sánh tổng với `i(i+1)/2`

**Từng đoạn:** giữ `total` = tổng các vị trí đã bật. Vì các vị trí phân biệt và dương, `total == i(i+1)/2` ⇔ tập vị trí là `{1..i}`. Python có số nguyên lớn tùy ý nên không lo tràn (ở C++/Java cần `long`: tổng tới ≈ `1,25·10^9`).

In [11]:
def count_sum(flips):
    total = ans = 0
    for i, pos in enumerate(flips, start=1):
        total += pos
        if total == i * (i + 1) // 2:
            ans += 1
    return ans

for flips, want in EXAMPLES:
    print(flips, "->", count_sum(flips), "(mong đợi", want, ")")

[3, 2, 4, 1, 5] -> 2 (mong đợi 2 )
[4, 1, 2, 3] -> 1 (mong đợi 1 )


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def numTimesAllBlue(self, flips: list[int]) -> int:
        total = ans = 0
        for i, pos in enumerate(flips, 1):
            total += pos
            if total == i * (i + 1) // 2:
                ans += 1
        return ans

print([Solution().numTimesAllBlue(f) for f, _ in EXAMPLES])

[2, 1]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Con trỏ "bit 0 đầu tiên" (khấu hao)

**Từng đoạn:** giữ mảng bit và con trỏ `p` = vị trí bit 0 nhỏ nhất. Sau khi bật một bit, nếu `bits[p] == 1` thì tiến `p` cho tới bit 0 kế tiếp. Chuỗi aligned sau bước `i` ⇔ `p == i + 1` (các bit `1..i` đều 1; và vì có đúng `i` bit 1 nên phần sau đều 0).

Mỗi vị trí bị `p` đi qua **đúng một lần** trong cả quá trình ⇒ tổng `O(n)` (khấu hao), dù có bước `p` nhảy xa.

In [13]:
def count_pointer(flips):
    n = len(flips)
    bits = [0] * (n + 2)          # thêm ô canh n+1 luôn = 0
    p = 1
    ans = 0
    for i, pos in enumerate(flips, start=1):
        bits[pos] = 1
        while bits[p]:
            p += 1
        if p == i + 1:
            ans += 1
    return ans

for flips, want in EXAMPLES:
    print(flips, "->", count_pointer(flips), "(mong đợi", want, ")")

[3, 2, 4, 1, 5] -> 2 (mong đợi 2 )
[4, 1, 2, 3] -> 1 (mong đợi 1 )


Code nộp LeetCode (Cách 4):

In [14]:
class Solution:
    def numTimesAllBlue(self, flips: list[int]) -> int:
        n = len(flips)
        on = [False] * (n + 2)
        p = 1
        ans = 0
        for i, pos in enumerate(flips, 1):
            on[pos] = True
            while on[p]:
                p += 1
            if p == i + 1:
                ans += 1
        return ans

print([Solution().numTimesAllBlue(f) for f, _ in EXAMPLES])

[2, 1]


**Độ phức tạp Cách 4:** `O(n)` thời gian khấu hao, `O(n)` bộ nhớ. Ưu điểm: cách này vẫn đúng ngay cả khi ta muốn biết "tiền tố toàn 1 dài nhất" tại mỗi thời điểm (thông tin giàu hơn chỉ có/không).

## 8. TRACE từng bước

Trace đồng thời ba đại lượng tóm tắt (max, tổng, con trỏ `p`) trên `flips = [3,2,4,1,5]`:

| i | bật | chuỗi | max | tổng | i(i+1)/2 | p | aligned |
|---|---|---|---|---|---|---|---|
| 1 | 3 | 00100 | 3 | 3 | 1 | 1 | ✗ |
| 2 | 2 | 01100 | 3 | 5 | 3 | 1 | ✗ |
| 3 | 4 | 01110 | 4 | 9 | 6 | 1 | ✗ |
| 4 | 1 | 11110 | 4 | 10 | 10 | 5 | ✓ |
| 5 | 5 | 11111 | 5 | 15 | 15 | 6 | ✓ |

In [15]:
def trace_aligned(flips):
    n = len(flips)
    bits = [0] * (n + 2)
    mx = total = ans = 0
    p = 1
    print(" i | bật | chuỗi      | max | tổng | i(i+1)/2 |  p | aligned | ans")
    for i, pos in enumerate(flips, start=1):
        bits[pos] = 1
        mx = max(mx, pos)
        total += pos
        while bits[p]:
            p += 1
        ok = mx == i
        assert ok == (total == i * (i + 1) // 2) == (p == i + 1)   # ba tiêu chí luôn trùng nhau
        ans += ok
        s = "".join(map(str, bits[1:n + 1]))
        print(f"{i:2d} | {pos:3d} | {s:10s} | {mx:3d} | {total:4d} | {i*(i+1)//2:8d} | {p:2d} | {'✓' if ok else '✗':7s} | {ans}")
    return ans

print("kết quả:", trace_aligned([3, 2, 4, 1, 5]))
print()
print("kết quả:", trace_aligned([4, 1, 2, 3]))

 i | bật | chuỗi      | max | tổng | i(i+1)/2 |  p | aligned | ans
 1 |   3 | 00100      |   3 |    3 |        1 |  1 | ✗       | 0
 2 |   2 | 01100      |   3 |    5 |        3 |  1 | ✗       | 0
 3 |   4 | 01110      |   4 |    9 |        6 |  1 | ✗       | 0
 4 |   1 | 11110      |   4 |   10 |       10 |  5 | ✓       | 1
 5 |   5 | 11111      |   5 |   15 |       15 |  6 | ✓       | 2
kết quả: 2

 i | bật | chuỗi      | max | tổng | i(i+1)/2 |  p | aligned | ans
 1 |   4 | 0001       |   4 |    4 |        1 |  1 | ✗       | 0
 2 |   1 | 1001       |   4 |    5 |        3 |  2 | ✗       | 0
 3 |   2 | 1101       |   4 |    7 |        6 |  3 | ✗       | 0
 4 |   3 | 1111       |   4 |   10 |       10 |  5 | ✓       | 1
kết quả: 1


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def numTimesAllBlue(self, flips: list[int]) -> int:
        mx = 0          # vị trí lớn nhất đã bật
        ans = 0
        for i, pos in enumerate(flips, 1):   # i: số bước (1-based)
            if pos > mx:
                mx = pos
            if mx == i:                      # i bit 1 đều nằm trong [1, i]
                ans += 1
        return ans

for flips, want in EXAMPLES:
    got = Solution().numTimesAllBlue(flips)
    print(flips, got, "OK" if got == want else "SAI")

[3, 2, 4, 1, 5] 2 OK
[4, 1, 2, 3] 1 OK


## 10. Vì sao thuật toán đúng

Sau bước `i`, tập vị trí đã bật là `S_i = {flips[0], ..., flips[i-1]}` gồm **đúng `i` số phân biệt** (vì `flips` là hoán vị).

- (⇒) Nếu chuỗi prefix-aligned thì `S_i = {1..i}` ⇒ `max(S_i) = i`.
- (⇐) Nếu `max(S_i) = i` thì `S_i ⊆ {1..i}`. Một tập con có `i` phần tử của tập `i` phần tử chính là toàn bộ ⇒ `S_i = {1..i}` ⇒ bit `1..i` đều 1 và các bit khác đều 0 ⇒ prefix-aligned.

Biến `mx` duy trì đúng `max(S_i)` (invariant cực đại tiền tố), nên đếm số `i` có `mx == i` là đáp án.

Với Cách 3: tổng của `i` số nguyên dương phân biệt, sắp tăng `a_1 < a_2 < ... < a_i`, có `a_k ≥ k` ⇒ tổng `≥ i(i+1)/2`, dấu bằng ⇔ `a_k = k` với mọi `k` ⇔ `S_i = {1..i}`.

## 11. Phân tích Time / Space

Gọi `n = len(flips)` (≤ `5·10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng + kiểm tra | `O(n²)` | `O(n)` |
| 2. Max tiền tố | `O(n)` | `O(1)` |
| 3. Tổng | `O(n)` | `O(1)` |
| 4. Con trỏ bit 0 đầu tiên | `O(n)` khấu hao | `O(n)` |

Không có khác biệt average/worst cho Cách 2–4. Cách 1 luôn `Θ(n²)` vì mỗi bước cắt lát và quét toàn bộ.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
METHODS = {
    "brute": count_brute,
    "max": count_max,
    "sum": count_sum,
    "pointer": count_pointer,
    "submit": lambda f: Solution().numTimesAllBlue(f),
}
edge = [
    ([1], 1),                       # n = 1
    ([1, 2, 3, 4, 5], 5),           # bật theo thứ tự: mọi bước đều aligned
    ([5, 4, 3, 2, 1], 1),           # ngược: chỉ bước cuối
    ([2, 1], 1),
    ([2, 1, 4, 3], 2),
    ([1, 3, 2, 4], 3),
    ([2, 1, 3, 5, 4], 3),
]
for name, fn in METHODS.items():
    for flips, want in EXAMPLES + edge:
        got = fn(flips)
        assert got == want, (name, flips, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 9 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định) — `flips` luôn là hoán vị hợp lệ

In [18]:
rng = random.Random(1375)
for _ in range(1000):
    n = rng.randint(1, 30)
    p = list(range(1, n + 1))
    rng.shuffle(p)
    res = {name: fn(p) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, (p, res)
print("1000 hoán vị ngẫu nhiên: mọi cách khớp nhau")

# kiểm tra vét cạn mọi hoán vị với n <= 6
from itertools import permutations
total = 0
for n in range(1, 7):
    for p in permutations(range(1, n + 1)):
        p = list(p)
        assert count_brute(p) == count_max(p) == count_sum(p) == count_pointer(p)
        total += 1
print("Vét cạn", total, "hoán vị (n <= 6): khớp")

1000 hoán vị ngẫu nhiên: mọi cách khớp nhau
Vét cạn 873 hoán vị (n <= 6): khớp


### 12.3 Test lớn theo ràng buộc (`n = 5·10^4`)

In [19]:
n = 50_000
big = list(range(1, n + 1)); random.Random(7).shuffle(big)
for name in ["max", "sum", "pointer", "submit"]:
    t0 = time.perf_counter()
    r = METHODS[name](big)
    print(f"{name:8s} -> {r} | {time.perf_counter() - t0:.4f}s")
print("thứ tự tăng:", count_max(list(range(1, n + 1))), "| thứ tự giảm:", count_max(list(range(n, 0, -1))))

max      -> 1 | 0.0030s
sum      -> 1 | 0.0047s
pointer  -> 1 | 0.0060s
submit   -> 1 | 0.0021s
thứ tự tăng: 50000 | thứ tự giảm: 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lẫn chỉ số 0-based với số bước 1-based

In [20]:
def wrong_zero_based(flips):
    mx = ans = 0
    for i, pos in enumerate(flips):    # SAI: i bắt đầu từ 0
        mx = max(mx, pos)
        if mx == i:
            ans += 1
    return ans

for flips, want in EXAMPLES + [([1, 2, 3], 3)]:
    print(flips, "-> sai:", wrong_zero_based(flips), "| đúng:", want)

[3, 2, 4, 1, 5] -> sai: 0 | đúng: 2
[4, 1, 2, 3] -> sai: 0 | đúng: 1
[1, 2, 3] -> sai: 0 | đúng: 3


**Vì sao sai:** `flips` chứa vị trí 1-based (`1..n`), còn `enumerate` mặc định đếm từ 0 ⇒ so `mx` với số bước bị lệch 1; `mx ≥ 1 > 0` ở bước đầu nên hầu như không bao giờ bằng. `[1,2,3]` cho 0 thay vì 3.

**Code đúng:** `enumerate(flips, start=1)` hoặc so `mx == i + 1`.

### Lỗi 2 — Chỉ kiểm tra bit vừa bật (`flips[i] == i`)

In [21]:
def wrong_current_only(flips):
    ans = 0
    for i, pos in enumerate(flips, start=1):
        if pos == i:                     # SAI: chỉ nhìn bước hiện tại
            ans += 1
    return ans

for flips, want in EXAMPLES + [([2, 1], 1), ([1, 3, 2], 2), ([3, 2, 1], 1)]:
    print(flips, "-> sai:", wrong_current_only(flips), "| đúng:", want)

[3, 2, 4, 1, 5] -> sai: 2 | đúng: 2
[4, 1, 2, 3] -> sai: 0 | đúng: 1
[2, 1] -> sai: 0 | đúng: 1
[1, 3, 2] -> sai: 1 | đúng: 2
[3, 2, 1] -> sai: 1 | đúng: 1


**Vì sao sai:** điều kiện phụ thuộc vào **toàn bộ** các bit đã bật, không chỉ bit vừa bật. `[2,1]`: sau bước 2 chuỗi là `11` (aligned) dù `flips[2] = 1 ≠ 2`. Ngược lại `[3,2,1]` ở bước 2 có `flips[2] = 2` nhưng chuỗi `011` không aligned (tổng cuối vô tình vẫn bằng 1 vì bỏ sót bước 3 — sai hai lần bù nhau); `[1,3,2]` lộ rõ: 1 thay vì 2.

**Code đúng:** dùng một đại lượng **tích lũy** (max tiền tố / tổng).

### Lỗi 3 — So sánh trước khi cập nhật max

In [22]:
def wrong_order(flips):
    mx = ans = 0
    for i, pos in enumerate(flips, start=1):
        if mx == i:                      # SAI: so với max của bước TRƯỚC
            ans += 1
        mx = max(mx, pos)
    return ans

for flips, want in EXAMPLES + [([1], 1)]:
    print(flips, "-> sai:", wrong_order(flips), "| đúng:", want)

[3, 2, 4, 1, 5] -> sai: 2 | đúng: 2
[4, 1, 2, 3] -> sai: 1 | đúng: 1
[1] -> sai: 0 | đúng: 1


**Vì sao sai:** trạng thái cần xét là **sau** khi bật bit ở bước `i`. So trước khi cập nhật nghĩa là đang so `max` của `i−1` bit với `i` — gần như luôn sai (`[1]` cho 0 thay vì 1).

**Code đúng:** cập nhật `mx` rồi mới so sánh.

### Lỗi 4 — Tính lại `max(flips[:i])` trong vòng lặp (đúng nhưng `O(n²)`)

Kết quả đúng nhưng chậm — ta đo thời gian và so với bản `O(n)` để thấy khác biệt:

In [23]:
def slow_recompute(flips):
    return sum(1 for i in range(1, len(flips) + 1) if max(flips[:i]) == i)   # O(n^2)

p = list(range(1, 8001)); random.Random(1).shuffle(p)
t0 = time.perf_counter(); r1 = slow_recompute(p); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = count_max(p); t2 = time.perf_counter() - t0
print(f"slow: {r1} trong {t1:.3f}s | O(n): {r2} trong {t2:.4f}s (n = 8000; n = 50000 sẽ chậm hơn ~40 lần)")

slow: 1 trong 0.394s | O(n): 1 trong 0.0005s (n = 8000; n = 50000 sẽ chậm hơn ~40 lần)


**Vì sao có hại:** `flips[:i]` tạo list mới dài `i` và `max` quét `i` phần tử ⇒ `O(n²)` thời gian và cấp phát bộ nhớ liên tục. Với `n = 5·10^4` có thể TLE.

**Code đúng:** giữ `mx` tích lũy.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mảng là hoán vị" + "sau `i` bước/`i` phần tử đầu có tạo thành `{1..i}`?"** ⇒ nghĩ ngay **max tiền tố == i** (hoặc tổng == i(i+1)/2).
- Cùng khuôn với **LeetCode 769 (Max Chunks To Make Sorted)**: chặt mảng tại `i` khi `max(arr[0..i]) == i`.
- Khi đề mô tả quá trình mô phỏng tốn kém, hãy tìm **đại lượng tóm tắt** cập nhật `O(1)` mỗi bước (max, min, tổng, đếm).
- Con trỏ đơn điệu chỉ tiến ("vị trí trống đầu tiên") ⇒ tổng chi phí khấu hao `O(n)`.
- Luôn tự kiểm tra 1-based/0-based bằng ví dụ `[1]` và `[1,2,3]`.

## 15. Biến thể của bài

1. **Suffix-aligned:** đếm bước mà bit `n−i+1..n` đều 1 ⇒ điều kiện `min == n − i + 1`.
2. **Trả về danh sách các bước** aligned thay vì số lượng.
3. **LeetCode 769 — Max Chunks To Make Sorted:** hoán vị `0..n−1`, số đoạn tối đa để sắp từng đoạn rồi nối lại được mảng tăng ⇒ đếm `i` có `max(arr[0..i]) == i`.
4. **LeetCode 768 — Max Chunks II** (có phần tử trùng): so `max tiền tố ≤ min hậu tố`.
5. **Tiền tố toàn 1 dài nhất sau mỗi bước** (không cần đúng bằng `i`): dùng Cách 4 (con trỏ `p − 1`).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | bật bit, kiểm tra cả chuỗi mỗi bước | `O(n²)` | `O(n)` | kiểm chứng, `n` nhỏ |
| 2. Max tiền tố | aligned ⇔ `max == i` | `O(n)` | `O(1)` | **lời giải chuẩn** |
| 3. Tổng | aligned ⇔ `tổng == i(i+1)/2` | `O(n)` | `O(1)` | cách nhìn khác, dễ chứng minh |
| 4. Con trỏ | aligned ⇔ bit 0 đầu tiên ở `i+1` | `O(n)` khấu hao | `O(n)` | khi cần thêm "độ dài tiền tố 1" |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `numTimesAllBlue([1,2,3,4])`, `numTimesAllBlue([4,3,2,1])`, `numTimesAllBlue([2,3,1,5,4,6])`?

**Bài 2 (tính tay):** lập bảng max tiền tố cho `flips = [2,1,3,5,4]` và cho biết ở những bước nào chuỗi aligned.

**Bài 3 (điền TODO):** trả về **danh sách các bước** (1-based) mà chuỗi aligned.

In [24]:
def aligned_steps(flips):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    steps = []
    mx = 0
    for i, pos in enumerate(flips, start=1):
        # TODO: cập nhật mx; nếu mx == i thì thêm i vào steps
        pass
    return steps

**Bài 4 (tự viết lại):** viết lại lời giải bằng **min hậu tố**: duyệt từ phải sang trái, đếm số `i` sao cho `min(flips[i..n−1]) > i` (chỉ số 0-based) — tức mọi vị trí bật **sau** bước `i` đều lớn hơn `i`. Kiểm chứng rằng kết quả trùng `count_max`.

**Bài 5 (mini challenge — LeetCode 769):** cho `arr` là hoán vị của `0..n−1`. Chia `arr` thành số đoạn **nhiều nhất** sao cho sắp xếp từng đoạn rồi nối lại được mảng tăng dần. Trả về số đoạn.

In [25]:
def max_chunks_to_sorted(arr):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Viết max tiền tố | tăng dần: mọi bước | 4, 1, 3 |
| 2 | max: 2,2,3,5,5 | so với i = 1..5 | aligned ở bước 2, 3, 5 |
| 3 | Giống Cách 2 | thay `ans += 1` bằng `append` | `steps.append(i)` |
| 4 | aligned sau bước `i+1` ⇔ các bước sau chỉ bật vị trí > `i+1` | duyệt ngược, giữ `mn` | đếm `i` (0-based) có `mn_suffix(i+1) > i+1`, cộng thêm bước cuối |
| 5 | Chỉ số 0-based: aligned ⇔ `max == i` | mỗi lần bằng là một đoạn kết thúc | `sum(max tiền tố == i)` |

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = [count_max(f) for f in ([1, 2, 3, 4], [4, 3, 2, 1], [2, 3, 1, 5, 4, 6])]
print("Bài 1:", ans1)
assert ans1 == [4, 1, 3]

# Bài 2
f = [2, 1, 3, 5, 4]
prefix_max, mx = [], 0
for x in f:
    mx = max(mx, x); prefix_max.append(mx)
steps2 = [i for i, m in enumerate(prefix_max, start=1) if m == i]
print("Bài 2: max tiền tố =", prefix_max, "| aligned ở bước", steps2)
assert prefix_max == [2, 2, 3, 5, 5] and steps2 == [2, 3, 5]

Bài 1: [4, 1, 3]
Bài 2: max tiền tố = [2, 2, 3, 5, 5] | aligned ở bước [2, 3, 5]


In [27]:
# Bài 3
def aligned_steps_sol(flips):
    steps, mx = [], 0
    for i, pos in enumerate(flips, start=1):
        mx = max(mx, pos)
        if mx == i:
            steps.append(i)
    return steps

assert aligned_steps_sol([3, 2, 4, 1, 5]) == [4, 5]
assert aligned_steps_sol([4, 1, 2, 3]) == [4]
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 20); p = list(range(1, n + 1)); rng.shuffle(p)
    assert len(aligned_steps_sol(p)) == count_brute(p)
print("Bài 3: OK")

Bài 3: OK


In [28]:
# Bài 4: min hậu tố
def count_suffix_min(flips):
    n = len(flips)
    ans = 1                       # sau bước n luôn aligned (mọi bit đều 1)
    mn = flips[-1]
    # xét bước s = i+1 (1-based) với i = n-2 .. 0: aligned <=> mọi vị trí bật ở bước > s đều > s
    for i in range(n - 2, -1, -1):
        s = i + 1
        if mn > s:
            ans += 1
        mn = min(mn, flips[i])
    return ans

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 25); p = list(range(1, n + 1)); rng.shuffle(p)
    assert count_suffix_min(p) == count_max(p), p
print("Bài 4: OK")

Bài 4: OK


In [29]:
# Bài 5
def max_chunks_to_sorted_sol(arr):
    mx = chunks = 0
    for i, x in enumerate(arr):          # arr là hoán vị 0..n-1 nên dùng 0-based
        mx = max(mx, x)
        if mx == i:
            chunks += 1
    return chunks

def chunks_brute(arr):
    # thử mọi cách chặt (2^(n-1)), kiểm tra sắp từng đoạn
    n, best = len(arr), 0
    for mask in range(1 << (n - 1)):
        out, start = [], 0
        for i in range(n):
            if i == n - 1 or mask >> i & 1:
                out += sorted(arr[start:i + 1]); start = i + 1
        if out == sorted(arr):
            best = max(best, bin(mask).count("1") + 1)
    return best

assert max_chunks_to_sorted_sol([4, 3, 2, 1, 0]) == 1
assert max_chunks_to_sorted_sol([1, 0, 2, 3, 4]) == 4
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 8); a = list(range(n)); rng.shuffle(a)
    assert max_chunks_to_sorted_sol(a) == chunks_brute(a), a
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tiền tố của hoán vị có lấp đầy {1..i}?":**

```python
mx = ans = 0
for i, x in enumerate(perm, 1):     # 1-based nếu perm chứa 1..n
    mx = max(mx, x)                 # cập nhật TRƯỚC
    if mx == i:                     # i số phân biệt, max = i  =>  đúng {1..i}
        ans += 1
return ans
```

(Hoán vị `0..n−1` ⇒ dùng `enumerate(perm)` và so `mx == i`.)

**Checklist trước khi submit:**
- [ ] Số bước 1-based khớp với giá trị 1-based trong `flips`.
- [ ] Cập nhật `mx` **trước** khi so sánh.
- [ ] Không tính lại `max(flips[:i])` trong vòng lặp.
- [ ] Thử: `[1]` → 1, `[1,2,3]` → 3, `[3,2,1]` → 1, `[2,1]` → 1.
- [ ] Đúng chữ ký: `numTimesAllBlue(self, flips: list[int]) -> int`.